# 6. The full cancer model, in your browser

Notebook 5 rebuilt the shape of the cancer model in sixty lines. This one runs the model itself:
[CARTopiaX](https://github.com/compiler-research/CARTopiaX), the agent-based model of CAR T-cells attacking a
solid tumour (after Luque et al., *Sci. Rep.* 2024), on the [BioDynaMo](https://biodynamo.org) simulation engine.
CARTopiaX's C++ sources and the part of BioDynaMo they use, 59 files in all, are compiled to WebAssembly into one
1.2 MB library.

Only this notebook needs the library, so it is not part of the kernel's start-up download: the next cell fetches it
and loads it into the running C++ kernel. The Python here calls into it through cppjit, as in the other notebooks.

What to expect: the browser gives the model one thread, and one step (0.1 simulated minutes) takes about a quarter
of a second, so the runs here are minutes of simulated time, not the paper's 30 days. With the same seed the
numbers are the same as the native single-thread build's, digit for digit.

In [ ]:
import os, time, numpy as np
t = time.time()
lazy.load_library("libcartopiax.so")    # the 1.2 MB WebAssembly library, fetched from this site once
cppjit.cppdef('extern "C" const char* cartopiax_simulate(double radius_um, int minutes, int csv_every_steps, int snapshot_every_steps, int seed);')
print(f"CARTopiaX loaded in {time.time() - t:.1f} s")

## A tumour organoid meets CAR T-cells

A spherical organoid of radius 60 µm (253 tumour cells) sits in a 1 mm box of tissue. Oxygen diffuses in from the
box faces and the tumour cells consume it; the tumour cells secrete an immunostimulatory factor. The model's
treatment schedule infuses 3,957 CAR T-cells at day 0 around the tumour. Every 20 steps the model writes a summary
row, every 50 steps a snapshot of all cells and of both substance fields.

The run takes about a minute; the model's own console output (its full parameter list) goes to `run.log`.

In [ ]:
minutes, seed = 20, 42
t = time.time()
run = str(cppjit.gbl.cartopiax_simulate(60.0, minutes, 20, 50, seed))
wall = time.time() - t
steps = int(minutes / 0.1)
print(f"{minutes} simulated minutes = {steps} steps in {wall:.0f} s ({1000 * wall / steps:.0f} ms per step); results in {run}/")

## What the model reports

The summary CSV is the model's own output (`output/final_data.csv`); a few of its 23 columns:

In [ ]:
csv = np.genfromtxt(f"{run}/output/final_data.csv", delimiter=",", names=True)
cols = ["total_minutes", "num_tumor_cells", "num_alive_cart", "average_oxygen_cancer_cells", "average_oxygen_all_cells"]
print("".join(f"{c:>30}" for c in cols))
for row in csv: print("".join(f"{row[c]:30.3f}" for c in cols))

## Looking inside

Each snapshot holds every cell (position, diameter, kind) and both substance fields on the model's 50³ grid (20 µm
voxels). Below: the cells in a 20 µm thick slab through the centre of the organoid at the last snapshot, over the
oxygen and the immunostimulatory factor in the same plane. The CAR T-cells start at least 50 µm from the tumour; in
20 minutes they have barely moved.

In [ ]:
import matplotlib.pyplot as plt
snaps = sorted(int(f[6:-4]) for f in os.listdir(run) if f.startswith("cells_"))
step = snaps[-1]
cells = np.fromfile(f"{run}/cells_{step}.f32", np.float32).reshape(-1, 6)   # x, y, z, diameter, kind, oncoprotein
res, lo, hi, box = open(f"{run}/oxygen.grid").read().split(); res, lo, hi = int(res), float(lo), float(hi)
def field(name, s): return np.fromfile(f"{run}/{name}_{s}.f32", np.float32).reshape(res, res, res)   # [z, y, x]
mid, view = res // 2, 200   # the grid's central plane; µm shown either side of the centre
slab = cells[(np.abs(cells[:, 2]) < 10) & (np.abs(cells[:, 0]) < view) & (np.abs(cells[:, 1]) < view)]
tumour, cart = slab[slab[:, 4] <= 5], slab[slab[:, 4] >= 6]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, name, label, cmap in ((axes[0], "oxygen", "oxygen (mmHg)", "inferno"), (axes[1], "immunostimulatory_factor", "immunostimulatory factor", "viridis")):
    im = ax.imshow(field(name, step)[mid], origin="lower", extent=(lo, hi, lo, hi), cmap=cmap)
    fig.colorbar(im, ax=ax, shrink=0.8, label=label)
    ax.scatter(tumour[:, 0], tumour[:, 1], s=6, c="#2b7a70", label=f"tumour cells ({len(tumour)} in view)")
    ax.scatter(cart[:, 0], cart[:, 1], s=6, c="#c9a6ff", label=f"CAR T-cells ({len(cart)} in view)")
    ax.set(xlim=(-view, view), ylim=(-view, view), xlabel="x (µm)", ylabel="y (µm)")
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle(f"CARTopiaX in the browser, minute {step * 0.1:.0f}: mid-plane slab of a 60 µm organoid")
plt.tight_layout(); plt.show()

The same two fields along a line through the centre of the organoid, one curve per snapshot: the oxygen dip settles
within the first snapshot (diffusion is fast at this scale), while the factor the tumour cells secrete keeps spreading outward.

In [ ]:
x = lo + (np.arange(res) + 0.5) * (hi - lo) / res
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for k, s in enumerate(snaps):
    colour = plt.cm.viridis(k / max(1, len(snaps) - 1))
    for ax, name in ((axes[0], "oxygen"), (axes[1], "immunostimulatory_factor")):
        ax.plot(x, field(name, s)[mid, mid], color=colour, label=f"minute {s * 0.1:.0f}")
axes[0].set(xlabel="x (µm)", ylabel="mmHg", xlim=(-300, 300), title="oxygen")
axes[1].set(xlabel="x (µm)", xlim=(-300, 300), title="immunostimulatory factor")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Try it

Change `minutes` or `seed` above and rerun from there; one simulated hour is 600 steps, a few minutes here.
For days of simulated time (the CAR T-cells finding and killing the tumour), run CARTopiaX natively, where it uses
all your cores; the setup is in the [CARTopiaX repository](https://github.com/compiler-research/CARTopiaX).

What the browser build leaves out: threads (GitHub Pages cannot enable WebAssembly threads), BioDynaMo's ROOT file
I/O and ParaView export (a small built-in adaptor writes the snapshots used above instead), and user-defined
random distributions. The model code is CARTopiaX's own, unchanged.